# Bayesian Optimisation Capstone — Week 2 Queries

This notebook builds directly on the Week 1 notebook. All changes from Week 1 are clearly marked with a `# CHANGED: ...` comment explaining what was updated and why.

### Summary of changes from Week 1
| Change | Week 1 | Week 2 | Reason |
|---|---|---|---|
| kappa (UCB trade-off) | Fixed at 2.576 for all | Adaptive: 1.5 / 2.0 / 2.5 per function | Results showed functions needed different balances |
| Candidate pool | 50,000 | 100,000 | More data accumulated, afford more search |
| Function 1 strategy | GP-UCB | Max-distance grid sweep | All outputs ~0, GP has no signal to learn from |
| Data source | Initial .npy files only | Initial + Week 1 query appended | Growing dataset |
| Seed | 42 | 99 | Avoid identical candidate draws to Week 1 |

## 1. Imports and Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern
from scipy.spatial.distance import cdist  # CHANGED: added for F1 max-distance strategy

# CHANGED: seed updated from 42 to 99 to avoid replicating Week 1 candidate draws
np.random.seed(99)

print('All libraries loaded.')

## 2. Load Initial Data and Append Week 1 Results

**CHANGED from Week 1:** We now append the query point and received output from Week 1 to each function's dataset before fitting the GP. This is how the dataset grows week by week.

In [ ]:
DATA_PATH = './data/'  # folder containing the initial .npy files

# CHANGED: Week 1 query points (submitted last round)
week1_queries = {
    1: np.array([0.020584, 0.969910]),
    2: np.array([0.813958, 0.968718]),
    3: np.array([0.412118, 0.126161, 0.493019]),
    4: np.array([0.390847, 0.455494, 0.413276, 0.470071]),
    5: np.array([0.342735, 0.792788, 0.998424, 0.935381]),
    6: np.array([0.690974, 0.164681, 0.103053, 0.037619, 0.154198]),
    7: np.array([0.030065, 0.557667, 0.280418, 0.193751, 0.310746, 0.755561]),
    8: np.array([0.190497, 0.131345, 0.026522, 0.042568, 0.730959, 0.283019, 0.005482, 0.388835])
}

# CHANGED: Week 1 outputs received from the portal
week1_outputs = {
    1: 1.966e-321,
    2: -0.017659539446735338,
    3: -0.03594378093624019,
    4: -1.3242014729441993,
    5: 2021.562348501267,
    6: -1.8227163856485316,
    7: 1.3096626182809736,
    8: 9.810010198309
}

functions = {}
for i in range(1, 9):
    X_init = np.load(f'{DATA_PATH}function_{i}_initial_inputs.npy')
    y_init = np.load(f'{DATA_PATH}function_{i}_initial_outputs.npy')

    # CHANGED: append Week 1 result to grow the dataset
    X = np.vstack([X_init, week1_queries[i]])
    y = np.append(y_init, week1_outputs[i])

    functions[i] = {'X': X, 'y': y}

# Summary
print(f"{'Function':<12} {'N pts':<8} {'Best Y':<14} {'Week1 Y':<16} {'Improved?'}")
print('-' * 65)
for i, d in functions.items():
    y_init = np.load(f'{DATA_PATH}function_{i}_initial_outputs.npy')
    improved = week1_outputs[i] > y_init.max()
    print(f"Function {i:<3} {len(d['y']):<8} {d['y'].max():<14.6f} {week1_outputs[i]:<16.6f} {'YES' if improved else 'no'}")

## 3. Feature-Output Correlation Analysis

**NEW in Week 2:** Before running the GP, we examine simple linear correlations between each input feature and the output. This does not replace the GP but provides a sanity check and informs interpretation.

In [ ]:
print("Feature-output Pearson correlations (linear relationship only):\n")
for func_id, d in functions.items():
    X, y = d['X'], d['y']
    corrs = [round(float(np.corrcoef(X[:, j], y)[0, 1]), 3) for j in range(X.shape[1])]
    labels = [f'x{j+1}:{c}' for j, c in enumerate(corrs)]
    print(f"F{func_id}: {', '.join(labels)}")

print("\nNote: correlations capture linear relationships only.")
print("Strong correlations (|r| > 0.4) suggest a feature is important, but nonlinearity may mean")
print("the GP will find patterns a linear model would miss.")

## 4. Core Functions: GP Fitting and Acquisition

The GP fitting function is unchanged from Week 1. The UCB function and query suggester have been updated with adaptive kappa support.

**CHANGED:** Added `max_distance_query()` for Function 1 where GP is uninformative.

In [ ]:
def fit_gp(X, y):
    """Unchanged from Week 1. Matern 5/2 kernel with 10 restarts."""
    kernel = Matern(nu=2.5)
    gp = GaussianProcessRegressor(
        kernel=kernel,
        n_restarts_optimizer=10,
        normalize_y=True
    )
    gp.fit(X, y)
    return gp


def ucb_acquisition(X_candidates, gp, kappa):
    """UCB score = mu + kappa * sigma. Unchanged logic, kappa now passed in."""
    mu, sigma = gp.predict(X_candidates, return_std=True)
    return mu + kappa * sigma


def suggest_next_query(X, y, n_dims, n_candidates=100000, kappa=2.0, seed=99):
    """
    CHANGED from Week 1:
      - n_candidates increased from 50,000 to 100,000
      - kappa is now passed in per function (was fixed at 2.576)
      - seed changed from 42 to 99
    """
    rng = np.random.default_rng(seed)
    gp = fit_gp(X, y)
    candidates = rng.uniform(0, 1, size=(n_candidates, n_dims))
    ucb_scores = ucb_acquisition(candidates, gp, kappa)
    best_idx = np.argmax(ucb_scores)
    best_x = candidates[best_idx]
    mu_b, sig_b = gp.predict(best_x.reshape(1, -1), return_std=True)
    return best_x, float(mu_b.ravel()[0]), float(sig_b.ravel()[0]), float(ucb_scores[best_idx])


# CHANGED: new function added for F1 — GP is useless with all-zero outputs
def max_distance_query(X_obs, resolution=100):
    """
    NEW in Week 2. Returns the grid point furthest from all observed inputs.
    Used for Function 1 where every output is ~0 and the GP has no signal.
    This ensures we probe a genuinely new region rather than guessing blindly.
    """
    x1 = np.linspace(0, 1, resolution)
    x2 = np.linspace(0, 1, resolution)
    X1, X2 = np.meshgrid(x1, x2)
    grid = np.column_stack([X1.ravel(), X2.ravel()])
    dists = cdist(grid, X_obs)          # distance from every grid point to every observed point
    min_dists = dists.min(axis=1)       # for each grid point, how close is the nearest observation?
    return grid[np.argmax(min_dists)]   # pick the grid point whose nearest neighbour is furthest away


def format_query(x):
    """Format submission string. Unchanged from Week 1."""
    return '-'.join([f'{v:.6f}' for v in x])


print('Functions defined.')

## 5. Adaptive Kappa Strategy

**CHANGED from Week 1:** kappa is now assigned per function based on Week 1 performance, rather than using a single value for all functions.

| kappa | Assigned to | Rationale |
|---|---|---|
| 1.5 | F5, F8 | Large improvements — shift toward exploitation |
| 2.0 | F3, F4, F7 | Mixed or small improvements — balanced |
| 2.5 | F2, F6 | Regressed in Week 1 — more exploration needed |
| N/A | F1 | Max-distance sweep, GP bypassed entirely |

In [ ]:
# CHANGED: kappa is now a dict, not a single float
kappa_per_function = {
    1: None,   # max-distance strategy, GP bypassed
    2: 2.5,    # regressed in Week 1 — explore more
    3: 2.0,    # marginal miss — balanced
    4: 2.0,    # improved, continue balanced
    5: 1.5,    # large improvement — exploit more
    6: 2.5,    # regressed — explore more
    7: 2.0,    # near best — balanced
    8: 1.5,    # improved — exploit more
}

strategy_notes = {
    1: "Max-distance grid sweep (no GP signal)",
    2: "GP-UCB kappa=2.5 (Week 1 regressed)",
    3: "GP-UCB kappa=2.0 (marginal miss)",
    4: "GP-UCB kappa=2.0 (improved, balanced)",
    5: "GP-UCB kappa=1.5 (large improvement, exploit)",
    6: "GP-UCB kappa=2.5 (Week 1 regressed)",
    7: "GP-UCB kappa=2.0 (near best, balanced)",
    8: "GP-UCB kappa=1.5 (improved, exploit)",
}
print('Kappa assignments confirmed.')

## 6. Run Week 2 Queries

In [ ]:
results = {}

for func_id, d in functions.items():
    X, y = d['X'], d['y']
    n_dims = X.shape[1]
    kappa = kappa_per_function[func_id]

    # CHANGED: Function 1 now uses max-distance strategy instead of GP-UCB
    if func_id == 1:
        best_x = max_distance_query(X)
        gp = fit_gp(X, y)
        mu_b = float(gp.predict(best_x.reshape(1, -1)).ravel()[0])
        sig_b = float(gp.predict(best_x.reshape(1, -1), return_std=True)[1].ravel()[0])
        ucb_val = float('nan')  # UCB not meaningful for F1 this round
    else:
        best_x, mu_b, sig_b, ucb_val = suggest_next_query(X, y, n_dims, kappa=kappa)

    query_str = format_query(best_x)

    results[func_id] = {
        'query_string': query_str,
        'best_x': best_x,
        'current_best_y': float(y.max()),
        'predicted_mu': mu_b,
        'predicted_sigma': sig_b,
        'ucb_score': ucb_val,
        'kappa': kappa,
        'strategy': strategy_notes[func_id],
    }

    print(f"Function {func_id} ({n_dims}D) | n={len(y)} | Best Y: {y.max():.6f}")
    print(f"  Strategy : {strategy_notes[func_id]}")
    print(f"  GP mu={mu_b:.4f} | sigma={sig_b:.4f}")
    print(f"  Query    : {query_str}")
    print()

## 7. Submission Strings

In [ ]:
print('=' * 60)
print('WEEK 2 SUBMISSION STRINGS')
print('=' * 60)
for func_id, r in results.items():
    print(f'Function {func_id}: {r["query_string"]}')
print('=' * 60)

## 8. GP Surface Visualisation for 2D Functions (F1 and F2)

**CHANGED for F1:** Now shows the max-distance query point overlaid on the GP surface, making it clear why the GP is not being trusted here (near-flat mean, inflated uncertainty everywhere).

In [ ]:
def plot_2d_gp(func_id, X, y, query_x, kappa=2.0, resolution=80, label='GP-UCB Query'):
    x1 = np.linspace(0, 1, resolution)
    x2 = np.linspace(0, 1, resolution)
    X1, X2 = np.meshgrid(x1, x2)
    grid = np.column_stack([X1.ravel(), X2.ravel()])

    gp = fit_gp(X, y)
    mu, sigma = gp.predict(grid, return_std=True)
    ucb = mu + (kappa or 2.0) * sigma

    mu_grid    = mu.reshape(resolution, resolution)
    sigma_grid = sigma.reshape(resolution, resolution)
    ucb_grid   = ucb.reshape(resolution, resolution)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    panels = [
        (mu_grid,    'GP Mean (mu)',    'RdYlGn'),
        (sigma_grid, 'GP Uncertainty',  'Blues'),
        (ucb_grid,   f'UCB (k={kappa or "N/A"})', 'plasma'),
    ]
    for ax, (data, title, cmap) in zip(axes, panels):
        im = ax.contourf(X1, X2, data, levels=30, cmap=cmap)
        plt.colorbar(im, ax=ax)
        ax.scatter(X[:-1, 0], X[:-1, 1], c='white', s=50, edgecolors='black',
                   linewidths=1, zorder=5, label='Initial data')
        # CHANGED: Week 1 query shown separately in a different colour
        ax.scatter(X[-1, 0], X[-1, 1], c='yellow', s=80, edgecolors='black',
                   linewidths=1, marker='D', zorder=6, label='Week 1 query')
        ax.scatter(query_x[0], query_x[1], c='red', s=150, marker='*',
                   edgecolors='black', linewidths=0.8, zorder=7, label=label)
        ax.set_title(title, fontweight='bold')
        ax.set_xlabel('x1'); ax.set_ylabel('x2')
        ax.legend(fontsize=7)

    plt.suptitle(f'Function {func_id} — Week 2 GP Surfaces', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'function_{func_id}_week2_gp.png', dpi=150, bbox_inches='tight')
    plt.show()


# F1: kappa=None triggers 'N/A' label, reflecting that UCB was not used
for func_id in [1, 2]:
    d = functions[func_id]
    r = results[func_id]
    label = 'Max-dist query' if func_id == 1 else 'GP-UCB query'
    plot_2d_gp(func_id, d['X'], d['y'], r['best_x'],
               kappa=kappa_per_function[func_id], label=label)

## 9. Summary Table

In [ ]:
rows = []
for func_id, r in results.items():
    rows.append({
        'Function': f'F{func_id}',
        'Dims': functions[func_id]['X'].shape[1],
        'N pts': len(functions[func_id]['y']),
        'Best Y': round(r['current_best_y'], 4),
        'kappa': r['kappa'],
        'GP mu': round(r['predicted_mu'], 4),
        'GP sigma': round(r['predicted_sigma'], 4),
        'Strategy': r['strategy'],
        'Query': r['query_string'],
    })

df = pd.DataFrame(rows).set_index('Function')
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 220)
print(df.to_string())

### Kappa schedule going forward
| Round | Default kappa | Notes |
|---|---|---|
| Week 1 | 2.576 | High exploration, very limited data |
| Week 2 | 1.5 / 2.0 / 2.5 | Adaptive per function |
| Week 3 | 1.0 / 1.5 / 2.0 | Continue decaying toward exploitation |
| Week 4+ | 0.5 / 1.0 | Mostly exploitation, converging on peaks |